In [ ]:
%matplotlib ipympl

import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
path = '/home/scratch1/bart/microhh/rcemip-II/thl_tests/short'

exps = ['thlD', 'thlE']
#exps = ['thlD', 'thlE', 'thlEs', 'thlEse']

def load(exp):
    f = f'{path}/{exp}/rcemip_ii.default.0000000.nc'
    root = xr.open_dataset(f, decode_times=False)
    thermo = xr.open_dataset(f, group='thermo', decode_times=False)
    return thermo.assign_coords(time=root.time/86400, z=root.z, zh=root.zh)

stats = {}
for exp in exps:
    stats[exp] = load(exp)

In [ ]:
names = ['T', 'ql', 'qr', 'ql_frac']
scales = [1, 1e6, 1e6, 100]
labels = ['T (K)', 'ql (mg/kg)', 'qr (mg/kg)', 'cloud fraction (%)']

means = {}
for exp, ds in stats.items():
    means[exp] = ds.sel(time=slice(float(ds.time[-1])-5, None)).mean('time')

z = means['thlD'].z / 1000

fig, axs = plt.subplots(2, 4, figsize=(10,6), sharey=True, layout='constrained')

for i, exp in enumerate(exps):
    for j in range(4):
        prof = means[exp][names[j]] * scales[j]
        axs[0,j].plot(prof, z, color=f'C{i}', label=exp)

        if exp != 'thlD':
            diff = prof - means['thlD'][names[j]] * scales[j]
            axs[1,j].plot(diff, z, color=f'C{i}', label=f'{exp}-thlD')

for j in range(4):
    axs[0,j].set_xlabel(labels[j])
    axs[1,j].set_xlabel(f'Δ{labels[j]}')

axs[0,0].set_ylabel('z (km)')
axs[1,0].set_ylabel('z (km)')
axs[0,0].legend()
axs[1,0].legend()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(8,6), sharex=True, layout='constrained')

for exp, ds in stats.items():
    axs[0].plot(ds.time, ds['ql_path']*1e3, label=exp)
    axs[1].plot(ds.time, ds['qr_path']*1e3)
    axs[2].plot(ds.time, ds['rr']*86400)

axs[0].set_ylabel('LWP (g/m2)')
axs[1].set_ylabel('RWP (g/m2)')
axs[2].set_ylabel('rr (mm/day)')
axs[2].set_xlabel('time (day)')
axs[0].legend()